In [1]:
from e3nn import o3
import torch

In [2]:
irrep = o3.Irreps('4x0e + 4x1o + 4x2e')

irrep

irrep.slices()

[slice(0, 4, None), slice(4, 16, None), slice(16, 36, None)]

In [3]:
from allegro.nn._strided._layout import StridedLayout

In [4]:
strided_irrep = StridedLayout(irrep, pad_to_multiple= 1)

In [5]:
# Seems like a major format. You can represent inexed in the form n m l
strided_irrep.indexes_to_strided.reshape((4, 9))

tensor([[ 0,  4,  5,  6, 16, 17, 18, 19, 20],
        [ 1,  7,  8,  9, 21, 22, 23, 24, 25],
        [ 2, 10, 11, 12, 26, 27, 28, 29, 30],
        [ 3, 13, 14, 15, 31, 32, 33, 34, 35]])

In [6]:
# transform back
strided_irrep.indexes_to_catted

tensor([ 0,  9, 18, 27,  1,  2,  3, 10, 11, 12, 19, 20, 21, 28, 29, 30,  4,  5,
         6,  7,  8, 13, 14, 15, 16, 17, 22, 23, 24, 25, 26, 31, 32, 33, 34, 35])

In [7]:
# this one is fine but do not set high spherical harmonics in right order
import numpy as np
np.array(range(36)).reshape((9, 4))

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19],
       [20, 21, 22, 23],
       [24, 25, 26, 27],
       [28, 29, 30, 31],
       [32, 33, 34, 35]])

In [8]:
instr = []
# Instructions
irreps_in1 = o3.Irreps('4x0e + 4x1o + 4x2e')
irreps_in2 = o3.Irreps('4x0e + 4x1o + 4x2e')
irreps_out = o3.Irreps('4x0e + 4x1o + 4x2e')


tmp_i_out: int = 0
for i_out, (_, ir_out) in enumerate(irreps_out):
    for i_1, (_, ir_in1) in enumerate(irreps_in1):
        for i_2, (_, ir_in2) in enumerate(irreps_in2):
            if ir_out in ir_in1 * ir_in2:
                instr.append((i_1, i_2, i_out))
                
                tmp_i_out += 1

In [9]:
instr

[(0, 0, 0),
 (1, 1, 0),
 (2, 2, 0),
 (0, 1, 1),
 (1, 0, 1),
 (1, 2, 1),
 (2, 1, 1),
 (0, 2, 2),
 (1, 1, 2),
 (2, 0, 2),
 (2, 2, 2)]

In [10]:
irreps_out[2]

4x2ee

In [11]:
# Example of two functions
#def codegen_strided_tensor_product_forward(
#    irreps_in1: o3.Irreps,
#    in1_var: List[float],
#    irreps_in2: o3.Irreps,
#    in2_var: List[float],
#    irreps_out: o3.Irreps,
#    out_var: List[float],
#    instructions: List[Instruction],
#    normalization: str = "component",
#    shared_weights: bool = False,
#    specialized_code: bool = True,
#    sparse_mode: Optional[str] = None,
#    pad_to_alignment: int = 1,
#) -> Optional[fx.GraphModule]:

#def Contracter(
#    irreps_in1,
#    irreps_in2,
#    irreps_out,
#    instructions: List[Tuple[int, int, int]],
#    has_weight: bool,
#    connection_mode: str,
#    pad_to_alignment: int = 1,
#    shared_weights: bool = False,
#    sparse_mode: Optional[str] = None,
#):

In [12]:
from e3nn.o3 import Instruction

layer_idx = 1

connection_mode=(
                    "uuu" if layer_idx > 0 or self.embed_initial_edge else "uvv"
                )
has_weight = True

instructions=[
            Instruction(
                i_in1,
                i_in2,
                i_out,
                connection_mode,
                has_weight,
                1.0,
                {
                    "uvw": (
                        irreps_in1[i_in1].mul,
                        irreps_in2[i_in2].mul,
                        irreps_out[i_out].mul,
                    ),
                    "uvu": (irreps_in1[i_in1].mul, irreps_in2[i_in2].mul),
                    "uvv": (irreps_in1[i_in1].mul, irreps_in2[i_in2].mul),
                    "uuw": (irreps_in1[i_in1].mul, irreps_out[i_out].mul),
                    "uuu": (irreps_in1[i_in1].mul,),
                    "uvuv": (
                        irreps_in1[i_in1].mul,
                        irreps_in2[i_in2].mul,
                    ),
                }[connection_mode],
            )
            for i_in1, i_in2, i_out in instr
        ]

In [13]:
# Gen big w3j

layout_in1 = StridedLayout(irreps_in1)
layout_in2 = StridedLayout(irreps_in2)
layout_out = StridedLayout(irreps_out)

w3j_values = []
w3j_index = []


for ins_i, ins in enumerate(instructions):
    mul_ir_in1 = layout_in1.base_irreps[ins.i_in1]
    mul_ir_in2 = layout_in2.base_irreps[ins.i_in2]
    mul_ir_out = layout_out.base_irreps[ins.i_out]
    
    
    this_w3j = o3.wigner_3j(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
    this_w3j_index = this_w3j.nonzero()
    w3j_values.append(
        this_w3j[this_w3j_index[:, 0], this_w3j_index[:, 1], this_w3j_index[:, 2]]
    )
    
    
    this_w3j_index[:, 0] += layout_in1.base_irreps[: ins.i_in1].dim
    this_w3j_index[:, 1] += layout_in2.base_irreps[: ins.i_in2].dim
    this_w3j_index[:, 2] += layout_out.base_irreps[: ins.i_out].dim    
    
    #print(mul_ir_out)
    
    
    # Now need to flatten the index to be for [pk][ij]
    w3j_index.append(
        torch.cat(
            (
                (ins_i if ins.has_weight else 0)  # unweighted all go in first path
                * layout_out.base_dim
                + this_w3j_index[:, 2].unsqueeze(-1),
                this_w3j_index[:, 0].unsqueeze(-1) * layout_in2.base_dim
                + this_w3j_index[:, 1].unsqueeze(-1),
            ),
            dim=1,
        )
    )

num_paths: int = len(instructions) if has_weight else 1
    
w3j = torch.sparse_coo_tensor(
        indices=torch.cat(w3j_index, dim=0).t(),
        values=torch.cat(w3j_values, dim=0),
        size=(
            num_paths * layout_out.base_dim,
            layout_in1.base_dim * layout_in2.base_dim,
        ),
    ).coalesce()

In [14]:
w3j_i_indexes = torch.div(
    w3j.indices()[1], layout_in1.base_dim, rounding_mode="floor"
)
w3j_j_indexes = w3j.indices()[1] % layout_in1.base_dim
w3j_is_ij_diagonal = (layout_in1.base_dim == layout_in2.base_dim) and torch.all(
    w3j_i_indexes == w3j_j_indexes
)

In [15]:
w3j_i_indexes

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 0, 0, 0, 1, 2, 3, 1, 1, 2, 3, 1, 2, 3, 1, 2,
        3, 3, 4, 5, 6, 8, 5, 6, 7, 4, 6, 7, 8, 0, 0, 0, 0, 0, 1, 3, 1, 2, 1, 2,
        3, 2, 3, 1, 3, 4, 5, 6, 7, 8, 4, 5, 6, 7, 4, 5, 5, 6, 7, 8, 4, 5, 6, 7,
        8, 4, 5, 6, 7, 7, 8, 5, 6, 7, 8])

In [16]:
w3j.dense_dim()

0

In [17]:
w3j_j_indexes

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 1, 2, 3, 0, 0, 0, 6, 8, 5, 4, 5, 6, 7, 4, 7,
        6, 8, 3, 2, 1, 1, 1, 2, 3, 1, 3, 2, 3, 4, 5, 6, 7, 8, 3, 1, 2, 1, 1, 2,
        3, 3, 2, 1, 3, 0, 0, 0, 0, 0, 6, 7, 4, 5, 7, 6, 8, 5, 4, 5, 4, 5, 6, 7,
        8, 5, 4, 7, 6, 8, 7, 5, 8, 7, 6])

In [18]:
w3j_is_ij_diagonal

tensor(False)

In [19]:
layout_in1.base_dim

9

In [20]:
kij_shape = (
                layout_out.base_dim,
                layout_in1.base_dim,
                layout_in2.base_dim,
            )
w3j = (
    w3j.to_dense()
    .reshape(((num_paths,) if num_paths > 1 else tuple()) + kij_shape)
.contiguous()
)

In [21]:
prod(irreps_in1)

NameError: name 'prod' is not defined

### Looking into the void

In [22]:
from typing import List, Optional, Tuple
from math import sqrt

import torch
from torch import fx

from e3nn import o3
from e3nn.util.jit import compile
from e3nn.util import prod
from e3nn.o3 import Instruction

from opt_einsum_fx import jitable, optimize_einsums_full

from allegro.nn._strided._layout import StridedLayout
from allegro.nn._strided._spmm import ExplicitGradSpmm


def codegen_strided_tensor_product_forward(
    irreps_in1: o3.Irreps,
    in1_var: List[float],
    irreps_in2: o3.Irreps,
    in2_var: List[float],
    irreps_out: o3.Irreps,
    out_var: List[float],
    instructions: List[Instruction],
    normalization: str = "component",
    shared_weights: bool = False,
    specialized_code: bool = True,
    sparse_mode: Optional[str] = None,
    pad_to_alignment: int = 1,
) -> Optional[fx.GraphModule]:
    """Returns None if strided doesn't make sense for this TP."""
    # TODO padding
    # Check if irreps can be strided
    try:
        layout_in1 = StridedLayout(irreps_in1, pad_to_multiple=pad_to_alignment)
        layout_in2 = StridedLayout(irreps_in2, pad_to_multiple=pad_to_alignment)
        layout_out = StridedLayout(irreps_out, pad_to_multiple=pad_to_alignment)
    except ValueError:
        # one cannot be strided
        return None

    # check the instructions
    assert specialized_code

    print("Irreps")
    print(irreps_in1, irreps_in2, irreps_out)
    print("Connection mode")
    print(instructions[0].connection_mode)
    print("Has weight")
    print(instructions[0].has_weight)
    print("Sparce mode")
    print(sparse_mode)
    print("Shared weights")
    print(shared_weights)
    
    connection_mode = instructions[0].connection_mode
    if not all(ins.connection_mode == connection_mode for ins in instructions):
        return None

    has_weight = instructions[0].has_weight
    if not all(ins.has_weight == has_weight for ins in instructions):
        return None
    if not has_weight:
        assert connection_mode == "uuu"  # for now

    # TODO: sort insturctions?

    # Make the big w3j
    w3j_index = []
    w3j_values = []

    for ins_i, ins in enumerate(instructions):
        mul_ir_in1 = layout_in1.base_irreps[ins.i_in1]
        mul_ir_in2 = layout_in2.base_irreps[ins.i_in2]
        mul_ir_out = layout_out.base_irreps[ins.i_out]

        assert mul_ir_in1.ir.p * mul_ir_in2.ir.p == mul_ir_out.ir.p
        assert (
            abs(mul_ir_in1.ir.l - mul_ir_in2.ir.l)
            <= mul_ir_out.ir.l
            <= mul_ir_in1.ir.l + mul_ir_in2.ir.l
        )

        if mul_ir_in1.dim == 0 or mul_ir_in2.dim == 0 or mul_ir_out.dim == 0:
            raise ValueError

        this_w3j = o3.wigner_3j(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
        this_w3j_index = this_w3j.nonzero()
        w3j_values.append(
            this_w3j[this_w3j_index[:, 0], this_w3j_index[:, 1], this_w3j_index[:, 2]]
        )

        # Normalize the path through its w3j entries
        # TODO: path_weight
        # TODO: in and out var
        if normalization == "component":
            w3j_norm_term = 2 * mul_ir_out.ir.l + 1
        if normalization == "norm":
            w3j_norm_term = (2 * mul_ir_in1.ir.l + 1) * (2 * mul_ir_in2.ir.l + 1)
        alpha = sqrt(
            ins.path_weight  # per-path weight
            * out_var[ins.i_out]  # enforce output variance
            * w3j_norm_term
            / sum(
                in1_var[i.i_in1]
                * in2_var[i.i_in2]
                * {
                    "uvw": (layout_in1.mul * layout_in2.mul),
                    "uvu": layout_in2.mul,
                    "uvv": layout_in1.mul,
                    "uuw": layout_in1.mul,
                    "uuu": 1,
                    "uvuv": 1,
                }[connection_mode]
                for i in instructions
                if i.i_out == ins.i_out
            )
        )
        w3j_values[-1].mul_(alpha)

        this_w3j_index[:, 0] += layout_in1.base_irreps[: ins.i_in1].dim
        this_w3j_index[:, 1] += layout_in2.base_irreps[: ins.i_in2].dim
        this_w3j_index[:, 2] += layout_out.base_irreps[: ins.i_out].dim
        # Now need to flatten the index to be for [pk][ij]
        w3j_index.append(
            torch.cat(
                (
                    (ins_i if ins.has_weight else 0)  # unweighted all go in first path
                    * layout_out.base_dim
                    + this_w3j_index[:, 2].unsqueeze(-1),
                    this_w3j_index[:, 0].unsqueeze(-1) * layout_in2.base_dim
                    + this_w3j_index[:, 1].unsqueeze(-1),
                ),
                dim=1,
            )
        )

    num_paths: int = len(instructions) if has_weight else 1

    w3j = torch.sparse_coo_tensor(
        indices=torch.cat(w3j_index, dim=0).t(),
        values=torch.cat(w3j_values, dim=0),
        size=(
            num_paths * layout_out.base_dim,
            layout_in1.base_dim * layout_in2.base_dim,
        ),
    ).coalesce()

    # w3j is k,i,j, so this is whether, for nonzero entries,
    # the i index is always equal to the j index. If so, then
    # it is diagonal and we can eliminate the j dimension
    # in this case we are only taking diagonal (i == j)
    # entries from the outer product; but those values are just
    # the direct multiplication of the two tensors, eliminating
    # the need for the outer product.
    # obviously this only makes sense if they have the same size as well
    # this is more or less a test of whether this TP is an inner product
    w3j_i_indexes = torch.div(
        w3j.indices()[1], layout_in1.base_dim, rounding_mode="floor"
    )
    w3j_j_indexes = w3j.indices()[1] % layout_in1.base_dim
    w3j_is_ij_diagonal = (layout_in1.base_dim == layout_in2.base_dim) and torch.all(
        w3j_i_indexes == w3j_j_indexes
    )
    if w3j_is_ij_diagonal:
        # change the w3j to eliminate the dimension
        # now its just k,i
        w3j = torch.sparse_coo_tensor(
            indices=torch.stack((w3j.indices()[0], w3j_i_indexes)),
            values=w3j.values(),
            size=(
                num_paths * layout_out.base_dim,
                layout_in1.base_dim,
            ),
        )
    
    print("Is ij diagonal")
    print(w3j_is_ij_diagonal)
    # TODO: support use of sparse w3j
    if sparse_mode is None:
        # in dense, must shape it for einsum:
        if w3j_is_ij_diagonal:
            kij_shape = (
                layout_out.base_dim,
                layout_in1.base_dim,
            )
        else:
            kij_shape = (
                layout_out.base_dim,
                layout_in1.base_dim,
                layout_in2.base_dim,
            )
        w3j = (
            w3j.to_dense()
            .reshape(((num_paths,) if num_paths > 1 else tuple()) + kij_shape)
            .contiguous()
        )
        del kij_shape
    elif sparse_mode == "coo":
        w3j = w3j.coalesce()
    elif sparse_mode == "csr":
        w3j = w3j.coalesce().to_sparse_csr()
    else:
        raise ValueError

    # Generate the mixer
    u, v, w = connection_mode
    uv = {"uv": "uv", "uu": "u"}[connection_mode[:2]]
    if has_weight:
        weight_label = {"uvw": "uvw", "uuu": "u", "uvv": "uv"}[connection_mode]

        z = "" if shared_weights else "z"

        weight_shape = {
            "uvw": (layout_in1.mul, layout_in2.mul, layout_out.mul),
            "uuu": (layout_in1.mul,),
            "uvv": (layout_in1.mul, layout_in2.mul),
        }[connection_mode]
        if num_paths > 1:
            # ^ if there's only one weighted path, the einsum simplifies without the p dimension
            weight_label = weight_label + "p"
            weight_shape = weight_shape + (num_paths,)
        if not shared_weights:
            weight_shape = (-1,) + weight_shape
    else:
        weight_shape = tuple()

    # generate actual code
    graph_out = fx.Graph()
    tracer = fx.proxy.GraphAppendingTracer(graph_out)

    def Proxy(n):
        return fx.Proxy(n, tracer=tracer)

    # = Function definitions =
    x1s_out = Proxy(graph_out.placeholder("x1", torch.Tensor))
    x2s_out = Proxy(graph_out.placeholder("x2", torch.Tensor))
    if has_weight:
        ws_out = Proxy(graph_out.placeholder("w", torch.Tensor))
        ws_out = ws_out.reshape(weight_shape)

    if sparse_mode is None:
        w3j_proxy = Proxy(graph_out.get_attr("_big_w3j"))

    # convert to strided
    x1s_out = x1s_out.reshape(-1, layout_in1.mul, layout_in1.base_dim)
    x2s_out = x2s_out.reshape(-1, layout_in2.mul, layout_in2.base_dim)

    # do the einsum
    # has shape zwk
    j = "i" if w3j_is_ij_diagonal else "j"
    ij = "i" if w3j_is_ij_diagonal else "ij"
    if has_weight:
        if sparse_mode is None:
            # use einsum for the full contract
            einstr = f"{z}{weight_label},z{u}i,z{v}{j},{'p' if num_paths > 1 else ''}k{ij}->z{w}k"
            out = torch.einsum(einstr, ws_out, x1s_out, x2s_out, w3j_proxy)
        else:
            outer = torch.einsum(f"z{u}i,z{v}{j}->z{uv}{ij}", x1s_out, x2s_out)
            # \/ has shape [pk][ij] * [ij][zuv] = [pk][zuv]
            contracted = Proxy(
                graph_out.call_module(
                    "_w3j_mm",
                    (
                        outer.reshape(
                            -1,
                            (
                                layout_in1.base_dim
                                if w3j_is_ij_diagonal
                                else layout_in1.base_dim * layout_in2.base_dim
                            ),
                        ).T.node,
                    ),
                )
            ).T.reshape(
                (-1,)
                + {"uv": (layout_in1.mul, layout_in2.mul), "uu": (layout_in1.mul,)}[
                    connection_mode[:2]
                ]
                + (num_paths, layout_out.base_dim)
            )
            out = torch.einsum(f"z{uv}pk,{z}{weight_label}->z{w}k", contracted, ws_out)
    else:
        if sparse_mode is None:
            # use einsum for the full contract
            einstr = f"z{u}i,z{v}{j},{'p' if num_paths > 1 else ''}k{ij}->z{w}k"
            out = torch.einsum(einstr, x1s_out, x2s_out, w3j_proxy)
        else:
            outer = torch.einsum(f"z{u}i,z{v}{j}->z{uv}{ij}", x1s_out, x2s_out)
            # \/ has shape [k][ij] * [ij][zuv] = [pk][zuv]
            out = Proxy(
                graph_out.call_module(
                    "_w3j_mm",
                    (
                        outer.reshape(
                            -1,
                            (
                                layout_in1.base_dim
                                if w3j_is_ij_diagonal
                                else layout_in1.base_dim * layout_in2.base_dim
                            ),
                        ).T.node,
                    ),
                )
            ).T.reshape(
                (
                    -1,
                    layout_in1.mul,  # its only uuu for now
                    layout_out.base_dim,
                )
            )

    graph_out.output(out.node)

    # check graphs
    graph_out.lint()

    # Make GraphModules
    # By putting the constants in a Module rather than a dict,
    # we force FX to copy them as buffers instead of as attributes.
    #
    # FX seems to have resolved this issue for dicts in 1.9, but we support all the way back to 1.8.0.
    constants_root = torch.nn.Module()
    constants_root.register_buffer("_big_w3j", w3j)
    if sparse_mode is not None:
        constants_root._w3j_mm = ExplicitGradSpmm(w3j)
    graphmod_out = fx.GraphModule(constants_root, graph_out, class_name="tp_forward")

    if True:  # optimize_einsums
        # Note that for our einsums, we can optimize _once_ for _any_ batch dimension
        # and still get the right path for _all_ batch dimensions.
        # This is because our einsums are essentially of the form:
        #    zuvw,ijk,zuvij->zwk    OR     uvw,ijk,zuvij->zwk
        # In the first case, all but one operands have the batch dimension
        #    => The first contraction gains the batch dimension
        #    => All following contractions have batch dimension
        #    => All possible contraction paths have cost that scales linearly in batch size
        #    => The optimal path is the same for all batch sizes
        # For the second case, this logic follows as long as the first contraction is not between the first two operands. Since those two operands do not share any indexes, contracting them first is a rare pathological case. See
        # https://github.com/dgasmith/opt_einsum/issues/158
        # for more details.
        #
        # TODO: consider the impact maximum intermediate result size on this logic
        #         \- this is the `memory_limit` option in opt_einsum
        # TODO: allow user to choose opt_einsum parameters?
        #
        # We use float32 and zeros to save memory and time, since opt_einsum_fx looks only at traced shapes, not values or dtypes.
        batchdim = 4
        example_inputs = (
            torch.zeros((batchdim, layout_in1.dim)),
            torch.zeros((batchdim, layout_in2.dim)),
            torch.zeros(
                1 if shared_weights else batchdim,
                sum(prod(ins.path_shape) for ins in instructions if ins.has_weight),
            ),
        )
        graphmod_out = jitable(optimize_einsums_full(graphmod_out, example_inputs))

    graphmod_out.weight_shape = weight_shape
    graphmod_out._dim_in1 = layout_in1.base_dim
    graphmod_out._dim_in2 = layout_in2.base_dim
    graphmod_out._dim_out = layout_out.base_dim
    graphmod_out._mul_out = layout_out.mul
    graphmod_out.weight_numel = abs(prod(weight_shape))

    return graphmod_out

In [23]:
from e3nn.o3 import wigner_3j

# Triangular ineguality for path existance
def tri_ineq(l1, l2, l3):
    return max([l1, l2, l3]) <= min([l1 + l2, l2 + l3, l1 + l3])

lmax = 10

tot_symb = 0.
tot_nonzero = 0.
for l3 in range (lmax+1):
    for l2 in range(lmax+1): 
        for l1 in range(lmax+1):
            tot_symb += (2*l1+1) * (2*l2+1) * (2*l3+1)
            if tri_ineq(l1, l2, l3):
                tot_nonzero += (wigner_3j(l1, l2, l3) != 0).sum()

In [24]:
# Total matrix sparcity
tot_nonzero / tot_symb

tensor(0.0704)

In [25]:
ir = o3.Irreps('4x0e + 4x1o + 4x2e')

o3.Irreps([el[1] for el in ir]).dim

9

In [26]:
def ETN_third_order_step_forward(
    irreps_in1: o3.Irreps,
    irreps_in2: o3.Irreps,
    irreps_out: o3.Irreps,
    instructions: List[Instruction],
) -> Optional[fx.GraphModule]:
    """Returns next feature vector"""

    # Make the big w3j
    w3j_index = []
    w3j_values = []


    base_in1 = o3.Irreps([el[1] for el in irreps_in1])
    base_in2 = o3.Irreps([el[1] for el in irreps_in2])
    base_out = o3.Irreps([el[1] for el in irreps_out])
    
    for ins_i, ins in enumerate(instructions):
        mul_ir_in1 = base_in1[ins.i_in1]
        mul_ir_in2 = base_in2[ins.i_in2]
        mul_ir_out = base_out[ins.i_out]

        assert mul_ir_in1.ir.p * mul_ir_in2.ir.p == mul_ir_out.ir.p
        assert (
            tri_ineq(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
        )

        if mul_ir_in1.dim == 0 or mul_ir_in2.dim == 0 or mul_ir_out.dim == 0:
            raise ValueError

        this_w3j = o3.wigner_3j(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
        this_w3j_index = this_w3j.nonzero()
        w3j_values.append(
            this_w3j[this_w3j_index[:, 0], this_w3j_index[:, 1], this_w3j_index[:, 2]]
        )

        
        this_w3j_index[:, 0] += base_in1[: ins.i_in1].dim
        this_w3j_index[:, 1] += base_in2[: ins.i_in2].dim
        this_w3j_index[:, 2] += base_out[: ins.i_out].dim
        # Now need to flatten the index to be for [pk][ij]
        w3j_index.append(
            torch.cat(
                (   this_w3j_index[:, 2].unsqueeze(-1),
                    this_w3j_index[:, 0].unsqueeze(-1) * base_in2.dim
                    + this_w3j_index[:, 1].unsqueeze(-1),
                ),
                dim=1,
            )
        )

    num_paths: int = len(instructions)

    w3j = torch.sparse_coo_tensor(
        indices=torch.cat(w3j_index, dim=0).t(),
        values=torch.cat(w3j_values, dim=0),
        size=(
            num_paths * base_out.dim,
            base_in1.dim * base_in2.dim,
        ),
    ).coalesce()

    w3j_i_indexes = torch.div(
        w3j.indices()[1], base_in1.dim, rounding_mode="floor"
    )
    w3j_j_indexes = w3j.indices()[1] % base_in1.dim
    
    # in dense, must shape it for einsum:
    kij_shape = (
        base_out.dim,
        base_in1.dim,
        base_in2.dim,
    )
    w3j = (
        w3j.to_dense()
        .reshape(((num_paths,) if num_paths > 1 else tuple()) + kij_shape)
        .contiguous()
    )

    # Generate the mixer
    C_shape = (irreps_in1[0].mul, irreps_in2[0].mul, irreps_out[0].mul) + (num_paths,)

    # generate actual code
    graph_out = fx.Graph()
    tracer = fx.proxy.GraphAppendingTracer(graph_out)

    def Proxy(n):
        return fx.Proxy(n, tracer=tracer)

    # = Function definitions =
    u_in = Proxy(graph_out.placeholder("u_in", torch.Tensor))
    F = Proxy(graph_out.placeholder("F", torch.Tensor))
    
    C = Proxy(graph_out.placeholder("C", torch.Tensor))
    C = C.reshape(C_shape)

    
    w3j_proxy = Proxy(graph_out.get_attr("_big_w3j"))

    # convert to strided
    u_in = u_in.reshape(-1, base_in1.dim, C_shape[0])
    F = F.reshape(-1, base_in2.dim, C_shape[1])

    # do the einsum
    
    einstr = f"uvwp,ziu,zjv,pkij->zkw"
    u_out = torch.einsum(einstr, C, u_in, F, w3j_proxy)
    
    graph_out.output(u_out.node)

    # check graphs
    graph_out.lint()

    # Make GraphModules
    # By putting the constants in a Module rather than a dict,
    # we force FX to copy them as buffers instead of as attributes.
    #
    # FX seems to have resolved this issue for dicts in 1.9, but we support all the way back to 1.8.0.
    constants_root = torch.nn.Module()
    constants_root.register_buffer("_big_w3j", w3j)
    
    graphmod_out = fx.GraphModule(constants_root, graph_out, class_name="etn_step_forward")

    if True:  # optimize_einsums
        # Note that for our einsums, we can optimize _once_ for _any_ batch dimension
        # and still get the right path for _all_ batch dimensions.
        # This is because our einsums are essentially of the form:
        #    zuvw,ijk,zuvij->zwk    OR     uvw,ijk,zuvij->zwk
        # In the first case, all but one operands have the batch dimension
        #    => The first contraction gains the batch dimension
        #    => All following contractions have batch dimension
        #    => All possible contraction paths have cost that scales linearly in batch size
        #    => The optimal path is the same for all batch sizes
        # For the second case, this logic follows as long as the first contraction is not between the first two operands. Since those two operands do not share any indexes, contracting them first is a rare pathological case. See
        # https://github.com/dgasmith/opt_einsum/issues/158
        # for more details.
        #
        # TODO: consider the impact maximum intermediate result size on this logic
        #         \- this is the `memory_limit` option in opt_einsum
        # TODO: allow user to choose opt_einsum parameters?
        #
        # We use float32 and zeros to save memory and time, since opt_einsum_fx looks only at traced shapes, not values or dtypes.
        batchdim = 4
        example_inputs = (
            torch.zeros((batchdim, irreps_in1.dim)),
            torch.zeros((batchdim, irreps_in2.dim)),
            torch.zeros(
                1,
                sum(prod(ins.path_shape) for ins in instructions),
            ),
        )
        graphmod_out = jitable(optimize_einsums_full(graphmod_out, example_inputs))

    graphmod_out.C_shape = C_shape
    graphmod_out._dim_in1 = base_in1.dim
    graphmod_out._dim_in2 = base_in2.dim
    graphmod_out._dim_out = base_out.dim
    graphmod_out._mul_out = irreps_out[0].mul
    graphmod_out.C_numel = abs(prod(C_shape))

    return graphmod_out


def Contracter_ETN(
    irreps_in1,
    irreps_in2,
    irreps_out,
):
    irreps_in1 = o3.Irreps(irreps_in1)
    assert all(mul == irreps_in1[0].mul for mul, ir in irreps_in1)
    irreps_in2 = o3.Irreps(irreps_in2)
    assert all(mul == irreps_in2[0].mul for mul, ir in irreps_in2)
    irreps_out = o3.Irreps(irreps_out)
    assert all(mul == irreps_out[0].mul for mul, ir in irreps_out)
    
    
    instructions: List[Tuple[int, int, int]] = []
    tmp_i_out: int = 0
    for i_out, (_, ir_out) in enumerate(irreps_out):
        for i_1, (_, ir_in1) in enumerate(irreps_in1):
            for i_2, (_, ir_in2) in enumerate(irreps_in2):
                if ir_out in ir_in1 * ir_in2:
                    instructions.append((i_1, i_2, i_out))

                    tmp_i_out += 1
    
    print(instr)
    connection_mode = 'uvw'
    has_weight = True
    
    
    
    mod = ETN_third_order_step_forward(
        irreps_in1,
        irreps_in2,
        irreps_out,
        instructions=[
            Instruction(
                i_in1,
                i_in2,
                i_out,
                connection_mode,
                has_weight,
                1.0,
                (
                        irreps_in1[i_in1].mul,
                        irreps_in2[i_in2].mul,
                        irreps_out[i_out].mul,
                ),
            )
            for i_in1, i_in2, i_out in instructions
        ]
    )

    mod = compile(mod)
    return mod

In [27]:
lmax = 2
Nc = 10
rank = 10

irreps_in1 = o3.Irreps([(rank, (i,(-1)**(i))) for i in range(lmax + 1)])
irreps_in2 = o3.Irreps([(Nc, (i,(-1)**(i))) for i in range(lmax + 1)])
irreps_out = o3.Irreps([(rank, (i,(-1)**(i))) for i in range(lmax + 1)])

tp = Contracter_ETN(
    irreps_in1,
    irreps_in2, 
    irreps_out
)

[(0, 0, 0), (1, 1, 0), (2, 2, 0), (0, 1, 1), (1, 0, 1), (1, 2, 1), (2, 1, 1), (0, 2, 2), (1, 1, 2), (2, 0, 2), (2, 2, 2)]


/home/vladimir/anaconda3/envs/spingnn/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(


### Test on real data

In [28]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config


default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cpu',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)

# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [29]:
config = Config.from_file('./configs/example_ETN.yaml', defaults=default_config)
    

dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

dataset[0]

Processing dataset...
Done!


AtomicData(atom_types=[21, 1], cell=[3, 3], edge_cell_shift=[364, 3], edge_index=[2, 364], forces=[21, 3], pbc=[3], pos=[21, 3], total_energy=[1])

In [30]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
Nc = 10 # number of chennels for F features from ETN paper
N_rank_spec = 4 # hidden rank of reduction for type radial tensor
config['Nc'] = Nc
config['N_rank_spec'] = N_rank_spec

# ETN parameters
config['d'] = 4 # dimention of the tensor train
config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

DEBUG:root:* Initialize Output
  ...generate file name results/aspirin/example/log
  ...open log file results/aspirin/example/log
  ...generate file name results/aspirin/example/metrics_epoch.csv
  ...open log file results/aspirin/example/metrics_epoch.csv
  ...generate file name results/aspirin/example/metrics_initialization.csv
  ...open log file results/aspirin/example/metrics_initialization.csv
  ...generate file name results/aspirin/example/metrics_batch_train.csv
  ...open log file results/aspirin/example/metrics_batch_train.csv
  ...generate file name results/aspirin/example/metrics_batch_val.csv
  ...open log file results/aspirin/example/metrics_batch_val.csv
  ...generate file name results/aspirin/example/best_model.pth
  ...generate file name results/aspirin/example/last_model.pth
  ...generate file name results/aspirin/example/trainer.pth
  ...generate file name results/aspirin/example/config.yaml
Torch device: cpu
instantiate Loss
...Loss_param = dict(
...   optional_args =

In [31]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[0])

In [32]:
data_new = final_model(data0)

In [33]:
F = data_new['node_features_F']

In [34]:
C = torch.nn.Parameter(torch.Tensor(10, 10, 10, 11))
torch.nn.init.kaiming_uniform_(C, a=math.sqrt(3))


F_out = tp(F, F, C)

### Check on equivariance

In [35]:

import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

data = data0


import copy

data_rot = {key: torch.clone(data0[key]) for key in data0}

irreps_sh = o3.Irreps('1x0e + 1x1o + 1x2e') #o3.Irreps.spherical_harmonics(lmax=2)
irreps_sh_r = o3.Irreps('1x1o')

alpha, beta, gamma = o3.rand_angles(100)

rot_matrix = irreps_sh.D_from_angles(alpha[0], beta[0], gamma[0])
rot_matrix_r = irreps_sh_r.D_from_angles(alpha[0], beta[0], gamma[0])


data_rot['pos'] = data_rot['pos'] @ rot_matrix_r

In [36]:
torch.manual_seed(32)

C = torch.nn.Parameter(torch.Tensor(10, 10, 10, 11))
torch.nn.init.kaiming_uniform_(C, a=math.sqrt(3))


data_out = final_model(data)

data_out_rot = final_model(data_rot)


F = data_out['node_features_F'] # features
F_out = tp(F, F, C)

F_rot = data_out_rot['node_features_F'] # features from rotated positions

F_out_rot = tp(F_rot, F_rot, C)

F_out_rot_rot = torch.einsum('Njn,jk->Nkn', F_out_rot, rot_matrix.T) # rotated features from rotated positions

assert torch.allclose(F_out, F_out_rot_rot, atol=1e-08)
print('F is equivariant')

F is equivariant


### Alternative ETN module

In [58]:
from typing import Optional, List
import math
import functools

import torch
from torch import nn

from e3nn import o3
from e3nn.util.jit import compile_mode

from nequip.data import AtomicDataDict # dict of base keys
from nequip.nn import GraphModuleMixin # base class for GNN (stores irreps in and out)

from allegro import _keys # just dict of additional keys

from e3nn.o3 import wigner_3j # wigner_3j matrixes for three spherical harmonics coupling



# Triangular ineguality for path existance
def tri_ineq(l1, l2, l3):
    return max([l1, l2, l3]) <= min([l1 + l2, l2 + l3, l1 + l3])


@compile_mode("script")
class ETN_Module_alt(nn.Module, GraphModuleMixin):
    def __init__(self,
                 d: int,
                 N_rank_ett: List[int], 
                 irreps_in=None,
                 out_field: str = AtomicDataDict.PER_ATOM_ENERGY_KEY):
        
        super().__init__()
        self.out_field = out_field
        
        
        self.d = d
        self.Nc = irreps_in[_keys.NODE_FEATURES_F][0][0]
        self.register_buffer("N_rank_ett", torch.as_tensor(N_rank_ett, dtype=torch.long))
        
        # set up irreps
        self._init_irreps(
            irreps_in=irreps_in,
            required_irreps_in=[
                _keys.NODE_FEATURES_F
            ],
            irreps_out={_keys.NODE_FEATURES_ETN: o3.Irreps(
                    [(self.Nc, ir) for _, ir in irreps_in[_keys.NODE_FEATURES_F] ]),
                        out_field: o3.Irreps([(1, (0, 1))])}
        )
        
        
        # Parameters of the network
        
        # tensors for atomic features encoding
        lmax = irreps_in[_keys.EDGE_FEATURES_F].lmax # maximum spherical harmonic
        self.lmax = lmax
        
        # Second order cores(first and last)
        self.core2_1 = torch.nn.Parameter(torch.Tensor(lmax+1, self.Nc, N_rank_ett[0]))
        self.core2_d = torch.nn.Parameter(torch.Tensor(lmax+1, N_rank_ett[-1], self.Nc))
        
        
        
        # Third order cores
        # Assume irreps does not change 
        base_in1 = o3.Irreps([el[1] for el in irreps_in[_keys.EDGE_FEATURES_F]])
        base_in2 = o3.Irreps([el[1] for el in irreps_in[_keys.EDGE_FEATURES_F]])
        base_out = o3.Irreps([el[1] for el in irreps_in[_keys.EDGE_FEATURES_F]])
        

        # Building instructions
        instructions: List[Tuple[int, int, int]] = []
        tmp_i_out: int = 0
        for i_out, (_, ir_out) in enumerate(base_out):
            for i_1, (_, ir_in1) in enumerate(base_in1):
                for i_2, (_, ir_in2) in enumerate(base_in2):
                    if tri_ineq(ir_out.l, ir_in1.l, ir_in2.l):
                        instructions.append((i_1, i_2, i_out))
        
                        tmp_i_out += 1

        self.instructions = instructions
        # building large w3j
        for i_in1, i_in2, i_out in instructions:
            mul_ir_in1 = base_in1[ins.i_in1]
            mul_ir_in2 = base_in2[ins.i_in2]
            mul_ir_out = base_out[ins.i_out]
    
            assert mul_ir_in1.ir.p * mul_ir_in2.ir.p == mul_ir_out.ir.p
            assert (
                tri_ineq(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
            )
    
            if mul_ir_in1.dim == 0 or mul_ir_in2.dim == 0 or mul_ir_out.dim == 0:
                raise ValueError
    
            this_w3j = o3.wigner_3j(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
            this_w3j_index = this_w3j.nonzero()
            w3j_values.append(
                this_w3j[this_w3j_index[:, 0], this_w3j_index[:, 1], this_w3j_index[:, 2]]
            )
    
            
            this_w3j_index[:, 0] += base_in1[: ins.i_in1].dim
            this_w3j_index[:, 1] += base_in2[: ins.i_in2].dim
            this_w3j_index[:, 2] += base_out[: ins.i_out].dim
            # Now need to flatten the index to be for [pk][ij]
            w3j_index.append(
                torch.cat(
                    (   this_w3j_index[:, 2].unsqueeze(-1),
                        this_w3j_index[:, 0].unsqueeze(-1) * base_in2.dim
                        + this_w3j_index[:, 1].unsqueeze(-1),
                    ),
                    dim=1,
                )
            )
    
        num_paths: int = len(instructions)
    
        w3j = torch.sparse_coo_tensor(
            indices=torch.cat(w3j_index, dim=0).t(),
            values=torch.cat(w3j_values, dim=0),
            size=(
                num_paths * base_out.dim,
                base_in1.dim * base_in2.dim,
            ),
        ).coalesce()
        
        # in dense, must shape it for einsum:
        kij_shape = (
            base_out.dim,
            base_in1.dim,
            base_in2.dim,
        )
        
        # store in sparce, make dence in runtime
        self.w3j = w3j
        self.w3j_shape = (num_paths,) + kij_shape
        
        # third order free parameters
        self.cores3 = [torch.nn.Parameter(torch.Tensor(N_rank_ett[r], self.Nc, N_rank_ett[r+1], num_paths)) for r in range(d - 2)] 
        
        self.reset_parameters()
        
    def forward(self, data: AtomicDataDict.Type) -> AtomicDataDict.Type:
        
        # Input features
        F = data[_keys.NODE_FEATURES_F]
        
        # Defining tensors for TorchScript
        u_out = torch.zeros((F.shape[0], F.shape[1], self.N_rank_ett[-1]), dtype=F.dtype,
            device=F.device) # temporary verctor output of etn
            
        
        data[_keys.NODE_FEATURES_ETN] = torch.zeros_like(F, dtype=F.dtype,
            device=F.device) # final feature output
        
        slices = self.irreps_in[AtomicDataDict.EDGE_ATTRS_KEY].slices() # slices over irreps

        # getting w3j in dense mode
        w3j = (
            self.w3j.to_dense()
            .reshape(self.w3j_shape)
            .contiguous()
        )   
        
        # First transform using second order tensors
        for i, slice in enumerate(slices):
            u_out[:, slice, :] = torch.einsum('ij,Nmj->Nmi', self.core2_d[i], F[:, slice, :])
        
        # Series third order tensors
        for i in range(self.d - 2 - 1, -1, -1):

            # big contruction
            einstr = f"uvwp,zjw,ziv,pkij->zku"
            u_out = torch.einsum(einstr, self.cores3[i], u_out, F, w3j)

        # Last transform using second order tensor
        for i, slice in enumerate(slices):
            data[_keys.NODE_FEATURES_ETN][:, slice, :] = torch.einsum('ij,Nmj->Nmi', self.core2_1[i], u_out[:, slice, :])
        
        
        # Reduction to scalar
        data[self.out_field] = ( data[_keys.NODE_FEATURES_ETN] * F ).sum(dim = (-2, -1))
        

        return data
    
    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.core2_1, a=math.sqrt(3))
        torch.nn.init.kaiming_uniform_(self.core2_d, a=math.sqrt(3))
        
        for core in self.cores3:
            torch.nn.init.kaiming_uniform_(core, a=math.sqrt(3))

In [59]:
ETN_alt = ETN_Module_alt(d = config['d'],
                         N_rank_ett = config['N_rank_ett'], 
                         irreps_in = final_model.irreps_out,
                         out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY)

In [60]:
data_new_new = ETN_alt(data_new)